In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetime as required
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
error_df['timestamp'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

# 1) metric.csv aggregations
metric_ts_range = pd.Series({
    'min_timestamp': metric_df['timestamp'].min(),
    'max_timestamp': metric_df['timestamp'].max()
})

metric_grouped = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'])
    .agg(
        count=('value', 'count'),
        min=('value', 'min'),
        max=('value', 'max'),
        mean=('value', 'mean'),
        p95=('value', lambda x: x.quantile(0.95))
    )
    .reset_index()
)

# Keep numeric columns compactly rounded
metric_grouped[['min','max','mean','p95']] = metric_grouped[['min','max','mean','p95']].round(6)
metric_summary = metric_grouped.sort_values('count', ascending=False).head(50)

# 2) trace.csv aggregations
trace_ts_range = pd.Series({
    'min_timestamp': trace_df['timestamp'].min(),
    'max_timestamp': trace_df['timestamp'].max()
})

trace_grouped = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'])
    .agg(
        count=('value', 'count'),
        min=('value', 'min'),
        max=('value', 'max'),
        mean=('value', 'mean'),
        p95=('value', lambda x: x.quantile(0.95))
    )
    .reset_index()
)

trace_grouped[['min','max','mean','p95']] = trace_grouped[['min','max','mean','p95']].round(6)
trace_summary = trace_grouped.sort_values('count', ascending=False).head(50)

# 3) log.csv aggregations
log_ts_range = pd.Series({
    'min_timestamp': log_df['timestamp'].min(),
    'max_timestamp': log_df['timestamp'].max()
})

log_grouped = (
    log_df
    .groupby(['cmdb_id', 'log_name'])
    .agg(
        count=('value', 'count'),
        min=('value', 'min'),
        max=('value', 'max'),
        mean=('value', 'mean'),
        p95=('value', lambda x: x.quantile(0.95))
    )
    .reset_index()
)

log_grouped[['min','max','mean','p95']] = log_grouped[['min','max','mean','p95']].round(6)
log_summary = log_grouped.sort_values('count', ascending=False).head(50)

# 4) error_logs.csv summary: count, unique cmdb_id list, sample up to 10 messages (with timestamps and cmdb_id)
error_count = len(error_df)
unique_cmdb_series = pd.Series(sorted(error_df['cmdb_id'].dropna().unique()), name='unique_cmdb_ids')
error_sample = (
    error_df
    .loc[:, ['timestamp', 'cmdb_id', 'message']]
    .sort_values('timestamp')
    .head(10)
    .reset_index(drop=True)
)

# Display compact outputs (variables only)
metric_summary, metric_ts_range, trace_summary, trace_ts_range, log_summary, log_ts_range, error_count, unique_cmdb_series, error_sample

```
Out[1]:
```
(metric_df.shape[0], trace_df.shape[0], log_df.shape[0], error_df.shape[0])
summary = (
    f"Dataset time range (UTC): {metric_ts_range['min_timestamp'].isoformat()} to {metric_ts_range['max_timestamp'].isoformat()}.\n\n"
    f"metric.csv: {metric_df.shape[0]} rows, {metric_grouped.shape[0]} (cmdb_id,kpi_name) groups. Showing top examples from the aggregated summary:\n"
    f" - {metric_summary.iloc[0]['cmdb_id']} / {metric_summary.iloc[0]['kpi_name']}: count={int(metric_summary.iloc[0]['count'])}, "
    f"min={metric_summary.iloc[0]['min']}, max={metric_summary.iloc[0]['max']}, mean={metric_summary.iloc[0]['mean']}, p95={metric_summary.iloc[0]['p95']}\n"
    f" - {metric_summary.iloc[1]['cmdb_id']} / {metric_summary.iloc[1]['kpi_name']}: count={int(metric_summary.iloc[1]['count'])}, "
    f"min={metric_summary.iloc[1]['min']}, max={metric_summary.iloc[1]['max']}, mean={metric_summary.iloc[1]['mean']}, p95={metric_summary.iloc[1]['p95']}\n\n"
    f"trace.csv: {trace_df.shape[0]} rows, {trace_grouped.shape[0]} (cmdb_id,trace_name) groups. Top examples:\n"
    f" - {trace_summary.iloc[0]['cmdb_id']} / {trace_summary.iloc[0]['trace_name']}: count={int(trace_summary.iloc[0]['count'])}, "
    f"min={trace_summary.iloc[0]['min']}, max={trace_summary.iloc[0]['max']}, mean={trace_summary.iloc[0]['mean']}, p95={trace_summary.iloc[0]['p95']}\n"
    f" - {trace_summary.iloc[1]['cmdb_id']} / {trace_summary.iloc[1]['trace_name']}: count={int(trace_summary.iloc[1]['count'])}, "
    f"min={trace_summary.iloc[1]['min']}, max={trace_summary.iloc[1]['max']}, mean={trace_summary.iloc[1]['mean']}, p95={trace_summary.iloc[1]['p95']}\n\n"
    f"log.csv: {log_df.shape[0]} rows, {log_grouped.shape[0]} (cmdb_id,log_name) groups. Observations:\n"
    f" - Many services have log.error_count = 0 across the sampled period.\n"
    f" - Example: {log_summary.iloc[0]['cmdb_id']} / {log_summary.iloc[0]['log_name']}: count={int(log_summary.iloc[0]['count'])}, "
    f"min={log_summary.iloc[0]['min']}, max={log_summary.iloc[0]['max']}, mean={log_summary.iloc[0]['mean']}, p95={log_summary.iloc[0]['p95']}\n\n"
    f"error_logs.csv: {len(error_df)} rows. Unique cmdb_id values: {sorted(error_df['cmdb_id'].dropna().unique().tolist())}. "
    f"Sample messages (up to 10): none (no rows) if empty.\n\n"
    f"Notes: Aggregations were computed globally per file and results limited to top 50 aggregated rows as requested."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                         cmdb_id    kpi_name  count           min           max          mean           p95
368           ts-voucher-service      socket     25  4.000000e+00  4.000000e+00  4.000000e+00  4.000000e+00
0    ts-admin-basic-info-service         cpu     25  3.268930e-01  4.159764e+00  7.034420e-01  1.418215e+00
1    ts-admin-basic-info-service      diskio     25  4.288709e+04  6.076675e+04  5.217790e+04  5.997585e+04
2    ts-admin-basic-info-service       error     25  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
3    ts-admin-basic-info-service  latency-50     25  1.710100e-02  1.790900e-02  1.753600e-02  1.788200e-02
..                           ...         ...    ...           ...           ...           ...           ...
309              ts-travel-mongo         cpu     25  6.617180e-01  7.810630e-01  7.175150e-01  7.770880e-01
310              ts-travel-mongo      diskio     25  2.935311e+05  4.366016e+05  3.619596e+05  4.181987e+05
311              ts-travel-mongo         mem     25  6.637391e+07  6.677893e+07  6.660361e+07  6.675977e+07
312              ts-travel-mongo      socket     25  5.000000e+00  6.000000e+00  5.218000e+00  6.000000e+00
73             ts-config-service  latency-90     25  8.987000e-03  9.295000e-03  9.145000e-03  9.272000e-03

[50 rows x 7 columns], min_timestamp   2024-02-20 16:51:00+00:00
max_timestamp   2024-02-20 17:15:00+00:00
dtype: datetime64[ns, UTC],                          cmdb_id                                         trace_name  count       min        max       mean        p95
783              ts-user-service                 trace.to_ts-user-service.row_count     25  4.000000  28.000000  16.400000  25.200000
0                           root  trace.to_ts-admin-basic-info-service.duration_...     25  0.010566   0.014153   0.011639   0.012662
1                           root  trace.to_ts-admin-basic-info-service.duration_p95     25  0.011660   0.022190   0.014442   0.018089
2                           root    trace.to_ts-admin-basic-info-service.error_rate     25  0.000000   0.000000   0.000000   0.000000
3                           root     trace.to_ts-admin-basic-info-service.row_count     25  5.000000  22.000000  18.120000  21.800000
..                           ...                                                ...    ...       ...        ...        ...        ...
73   ts-admin-basic-info-service  trace.to_ts-admin-basic-info-service.duration_p95     25  0.007483   0.018222   0.009287   0.013239
74   ts-admin-basic-info-service    trace.to_ts-admin-basic-info-service.error_rate     25  0.000000   0.000000   0.000000   0.000000
75   ts-admin-basic-info-service     trace.to_ts-admin-basic-info-service.row_count     25  5.000000  22.000000  18.120000  21.800000
76   ts-admin-basic-info-service           trace.to_ts-config-service.duration_mean     25  0.003227   0.004639   0.003586   0.003806
77   ts-admin-basic-info-service            trace.to_ts-config-service.duration_p95     25  0.003354   0.008279   0.004172   0.004534

[50 rows x 7 columns], min_timestamp   2024-02-20 16:51:00+00:00
max_timestamp   2024-02-20 17:15:00+00:00
dtype: datetime64[ns, UTC],                         cmdb_id         log_name  count  min  max   mean   p95
0   ts-admin-basic-info-service  log.error_count     25    0    0   0.00   0.0
1   ts-admin-basic-info-service    log.row_count     25    7   33  27.16  32.6
2       ts-admin-travel-service  log.error_count     25    0    0   0.00   0.0
3       ts-admin-travel-service    log.row_count     25    9  140  31.64  35.2
7          ts-assurance-service    log.row_count     25    2   23  12.56  19.4
..                          ...              ...    ...  ...  ...    ...   ...
8                 ts-auth-mongo  log.error_count     24    0    0   0.00   0.0
26            ts-food-map-mongo  log.error_count     24    0    0   0.00   0.0
27            ts-food-map-mongo    log.row_count     24    1    1   1.00   1.0
23            ts-contacts-mongo    log.row_count     24    1    1   1.00   1.0
22            ts-contacts-mongo  log.error_count     24    0    0   0.00   0.0

[50 rows x 7 columns], min_timestamp   2024-02-20 16:51:00+00:00
max_timestamp   2024-02-20 17:15:00+00:00
dtype: datetime64[ns, UTC], 0, Series([], Name: unique_cmdb_ids, dtype: object), Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

# Reuse existing metric_df and metric_grouped (which contains global p95 per (cmdb_id, kpi_name))
# Prepare global p95 lookup
global_p95_df = metric_grouped[['cmdb_id', 'kpi_name', 'p95']].rename(columns={'p95': 'global_p95'})

# Incident window (UTC)
start = pd.to_datetime('2024-02-20 16:48:43', utc=True)
end = pd.to_datetime('2024-02-20 17:18:43', utc=True)

# Filter metric rows to the incident window
metric_window = metric_df[(metric_df['timestamp'] >= start) & (metric_df['timestamp'] <= end)].copy()

# Merge global p95 into the windowed rows (global threshold computed from full series)
metric_window = metric_window.merge(global_p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# Mark exceedances: value > global_p95 (strictly greater)
metric_window['exceed'] = metric_window['value'] > metric_window['global_p95']

# Aggregate per (cmdb_id, kpi_name)
agg = (
    metric_window
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .agg(
        group_count=('value', 'size'),
        exceed_count=('exceed', 'sum'),
        earliest_exceed_timestamp=('timestamp', lambda x: x[metric_window.loc[x.index, 'exceed']].min() if x.size>0 else pd.NaT),
        max_value=('value', 'max'),
        global_p95=('global_p95', 'first')
    )
    .reset_index()
)

# Compute exceed_ratio and format earliest_exceed_timestamp as ISO UTC strings
agg['exceed_ratio'] = agg['exceed_count'] / agg['group_count']
agg['earliest_exceed_timestamp'] = agg['earliest_exceed_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

# Exclude groups with group_count == 0 (shouldn't exist because grouped from metric_window) and sort
result = (
    agg[agg['group_count'] > 0]
    .sort_values(['exceed_count', 'exceed_ratio'], ascending=[False, False])
    .loc[:, ['cmdb_id', 'kpi_name', 'group_count', 'exceed_count', 'exceed_ratio', 'earliest_exceed_timestamp', 'max_value', 'global_p95']]
)

# Round numeric columns for compact display
result['exceed_ratio'] = result['exceed_ratio'].round(4)
result['max_value'] = result['max_value'].round(6)
result['global_p95'] = result['global_p95'].round(6)

# Limit to top 20 rows
window_summary = result.head(20)

# Display the summary
window_summary

```
Out[2]:
```
# Build a concise plain-English summary using the previously computed window_summary and incident window
top = window_summary.head(10)
total_groups = window_summary.shape[0]
summary_lines = []
summary_lines.append(f"Incident window (UTC): {start.isoformat()} to {end.isoformat()}.")
summary_lines.append(f"Analyzed (cmdb_id, kpi_name) groups with data in the window: {total_groups}.")
summary_lines.append("Top groups by exceed_count (value > global P95):")

for i, row in top.reset_index(drop=True).iterrows():
    summary_lines.append(
        f" {i+1}. {row['cmdb_id']} / {row['kpi_name']}: "
        f"group_count={int(row['group_count'])}, exceed_count={int(row['exceed_count'])}, "
        f"exceed_ratio={row['exceed_ratio']:.3f}, earliest_exceed={row['earliest_exceed_timestamp']}, "
        f"max_value={row['max_value']}, global_p95={row['global_p95']}"
    )

summary_lines.append("")
summary_lines.append(
    "Interpretation: The highest observed exceedances are for ts-admin-travel-service (workload) "
    "and ts-ticketinfo-service (latency-50) with 3 points above their global P95s within the window. "
    "Multiple ts-admin-basic-info-service KPIs (cpu, diskio, latency, mem, socket, workload) have 2 exceedances each. "
    "These groups warrant further investigation as they show values above their historical (global) 95th-percentile thresholds during the incident window."
)

summary = "\n".join(summary_lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  group_count  exceed_count  exceed_ratio earliest_exceed_timestamp     max_value    global_p95
23       ts-admin-travel-service    workload           25             3        0.1200      2024-02-20T16:51:00Z  3.842940e-01  3.667330e-01
292        ts-ticketinfo-service  latency-50           25             3        0.1200      2024-02-20T17:04:00Z  1.875500e-02  1.874700e-02
208    ts-preserve-other-service  latency-50           24             2        0.0833      2024-02-20T17:10:00Z  1.472500e+00  1.156047e+00
209    ts-preserve-other-service  latency-90           24             2        0.0833      2024-02-20T17:10:00Z  2.269500e+00  2.177051e+00
0    ts-admin-basic-info-service         cpu           25             2        0.0800      2024-02-20T16:55:00Z  4.159764e+00  1.418215e+00
1    ts-admin-basic-info-service      diskio           25             2        0.0800      2024-02-20T16:54:00Z  6.076675e+04  5.997585e+04
3    ts-admin-basic-info-service  latency-50           25             2        0.0800      2024-02-20T17:01:00Z  1.790900e-02  1.788200e-02
4    ts-admin-basic-info-service  latency-90           25             2        0.0800      2024-02-20T17:01:00Z  2.423600e-02  2.420200e-02
5    ts-admin-basic-info-service         mem           25             2        0.0800      2024-02-20T17:14:00Z  2.366293e+08  2.365564e+08
6    ts-admin-basic-info-service      socket           25             2        0.0800      2024-02-20T16:51:00Z  8.411765e+00  8.130000e+00
7    ts-admin-basic-info-service    workload           25             2        0.0800      2024-02-20T16:59:00Z  7.701000e-01  6.855230e-01
8         ts-admin-order-service         cpu           25             2        0.0800      2024-02-20T16:58:00Z  1.245400e-01  1.171390e-01
9         ts-admin-order-service      diskio           25             2        0.0800      2024-02-20T17:05:00Z  4.532119e+04  4.483762e+04
10        ts-admin-order-service         mem           25             2        0.0800      2024-02-20T17:08:00Z  1.960484e+08  1.960480e+08
11        ts-admin-order-service      socket           25             2        0.0800      2024-02-20T17:10:00Z  3.316667e+00  3.273030e+00
12        ts-admin-route-service         cpu           25             2        0.0800      2024-02-20T16:58:00Z  1.328890e-01  1.214310e-01
13        ts-admin-route-service      diskio           25             2        0.0800      2024-02-20T17:12:00Z  6.423014e+04  6.202917e+04
14        ts-admin-route-service         mem           25             2        0.0800      2024-02-20T17:06:00Z  1.887543e+08  1.887541e+08
15        ts-admin-route-service      socket           25             2        0.0800      2024-02-20T17:08:00Z  3.483333e+00  3.326667e+00
16       ts-admin-travel-service         cpu           25             2        0.0800      2024-02-20T16:52:00Z  4.799810e-01  4.575340e-01```
```

In [ ]:
# In[3]:

# Reuse existing trace_df and trace_grouped (trace_grouped contains global p95 per (cmdb_id, trace_name))
# Prepare global p95 lookup (ensure thresholds computed from full series)
global_p95_trace = trace_grouped[['cmdb_id', 'trace_name', 'p95']].rename(columns={'p95': 'global_p95'})

# Incident window (reuse start/end defined earlier)
# start, end already defined as:
# start = pd.to_datetime('2024-02-20 16:48:43', utc=True)
# end   = pd.to_datetime('2024-02-20 17:18:43', utc=True)

# Filter trace rows to the incident window
trace_window = trace_df[(trace_df['timestamp'] >= start) & (trace_df['timestamp'] <= end)].copy()

# Merge global p95 into windowed rows (global thresholds computed from full series)
trace_window = trace_window.merge(global_p95_trace, on=['cmdb_id', 'trace_name'], how='left')

# Mark exceedances: value > global_p95 (strictly greater)
trace_window['exceed'] = trace_window['value'] > trace_window['global_p95']

# Aggregate per (cmdb_id, trace_name)
agg_trace = (
    trace_window
    .groupby(['cmdb_id', 'trace_name'], dropna=False)
    .agg(
        group_count=('value', 'size'),
        exceed_count=('exceed', 'sum'),
        earliest_exceed_timestamp=('timestamp', lambda x: x[trace_window.loc[x.index, 'exceed']].min() if x.size>0 else pd.NaT),
        max_value=('value', 'max'),
        global_p95=('global_p95', 'first')
    )
    .reset_index()
)

# Compute exceed_ratio and format earliest_exceed_timestamp as ISO UTC strings
agg_trace['exceed_ratio'] = agg_trace['exceed_count'] / agg_trace['group_count']
agg_trace['earliest_exceed_timestamp'] = agg_trace['earliest_exceed_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

# Exclude groups with group_count == 0 (should not appear) and sort by exceed_count desc then exceed_ratio desc
trace_window_summary = (
    agg_trace[agg_trace['group_count'] > 0]
    .sort_values(['exceed_count', 'exceed_ratio'], ascending=[False, False])
    .loc[:, ['cmdb_id', 'trace_name', 'group_count', 'exceed_count', 'exceed_ratio', 'earliest_exceed_timestamp', 'max_value', 'global_p95']]
)

# Round numeric columns for compact display and limit to top 20 rows
trace_window_summary['exceed_ratio'] = trace_window_summary['exceed_ratio'].round(4)
trace_window_summary['max_value'] = trace_window_summary['max_value'].round(6)
trace_window_summary['global_p95'] = trace_window_summary['global_p95'].round(6)

trace_window_summary.head(20)

```
Out[3]:
```
# Build a concise plain-English summary from the previously computed trace_window_summary
n_groups = trace_window_summary.shape[0]
earliest_exceed_overall = trace_window_summary['earliest_exceed_timestamp'].min()
# top by exceed_count then exceed_ratio already; show top 6 examples
top_examples = trace_window_summary.head(6).reset_index(drop=True)

# Find most severe by max_value relative to global_p95 (ratio)
trace_window_summary['severity_ratio'] = trace_window_summary['max_value'] / trace_window_summary['global_p95']
most_severe = trace_window_summary.sort_values('severity_ratio', ascending=False).head(3).reset_index(drop=True)

summary_lines = []
summary_lines.append(f"Incident window (UTC): {start.isoformat()} to {end.isoformat()}.")
summary_lines.append(f"Trace groups with data in window: {n_groups}. Earliest exceedance observed at {earliest_exceed_overall}.")
summary_lines.append("")
summary_lines.append("Key findings:")
summary_lines.append("- Multiple trace relations involving ts-preserve-other-service show exceedances above their historical global P95 (each with 2 exceedances, group_count=21, exceed_ratio≈0.095).")
summary_lines.append("- Affected relations include preserve->ticketinfo, preserve->order-other, preserve->contacts, preserve->seat, preserve->station, preserve->user, and the reciprocal 'from_ts-preserve-other-service' entries in those services.")
summary_lines.append("")
summary_lines.append("Most notable elevated latencies (examples):")
for i, row in most_severe.iterrows():
    summary_lines.append(
        f" {i+1}. {row['cmdb_id']} / {row['trace_name']}: group_count={int(row['group_count'])}, "
        f"exceed_count={int(row['exceed_count'])}, earliest_exceed={row['earliest_exceed_timestamp']}, "
        f"max_value={row['max_value']}, global_p95={row['global_p95']}, severity_ratio={row['severity_ratio']:.2f}"
    )

summary_lines.append("")
summary_lines.append("Interpretation and recommendation:")
summary_lines.append("- The pattern points to ts-preserve-other-service interactions as the likely area of impact: several downstream and upstream traces around ~17:09–17:12 UTC exceed their global P95s (notably preserve->ticketinfo).")
summary_lines.append("- Although exceed_count is small (2 samples) and exceed_ratio is modest (~9.5%), the absolute increases (e.g., ticketinfo duration_p95 max 0.776s vs global_p95 0.481s) are meaningful and warrant investigation of ts-preserve-other-service and its calls to ticketinfo and order-other around 17:09–17:12 UTC.")
summary_lines.append("- Next steps: inspect ts-preserve-other-service logs, resource metrics, and network for that timeframe; correlate with service traces shown above.")

summary = "\n".join(summary_lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                         trace_name  group_count  exceed_count  exceed_ratio earliest_exceed_timestamp  max_value  global_p95
220        ts-contacts-service  trace.from_ts-preserve-other-service.duration_...           21             2        0.0952      2024-02-20T16:58:00Z   0.019722    0.012614
312     ts-order-other-service  trace.from_ts-preserve-other-service.duration_...           21             2        0.0952      2024-02-20T17:12:00Z   0.025108    0.016139
313     ts-order-other-service  trace.from_ts-preserve-other-service.duration_p95           21             2        0.0952      2024-02-20T17:12:00Z   0.064344    0.023464
396  ts-preserve-other-service         trace.to_ts-contacts-service.duration_mean           21             2        0.0952      2024-02-20T16:58:00Z   0.019722    0.012614
404  ts-preserve-other-service      trace.to_ts-order-other-service.duration_mean           21             2        0.0952      2024-02-20T17:12:00Z   0.025108    0.016139
405  ts-preserve-other-service       trace.to_ts-order-other-service.duration_p95           21             2        0.0952      2024-02-20T17:12:00Z   0.064344    0.023464
413  ts-preserve-other-service              trace.to_ts-seat-service.duration_p95           21             2        0.0952      2024-02-20T17:06:00Z   0.437964    0.417707
420  ts-preserve-other-service          trace.to_ts-station-service.duration_mean           21             2        0.0952      2024-02-20T17:02:00Z   0.003783    0.003579
421  ts-preserve-other-service           trace.to_ts-station-service.duration_p95           21             2        0.0952      2024-02-20T16:52:00Z   0.006862    0.006209
424  ts-preserve-other-service       trace.to_ts-ticketinfo-service.duration_mean           21             2        0.0952      2024-02-20T17:09:00Z   0.357564    0.211772
425  ts-preserve-other-service        trace.to_ts-ticketinfo-service.duration_p95           21             2        0.0952      2024-02-20T17:09:00Z   0.776073    0.481264
432  ts-preserve-other-service             trace.to_ts-user-service.duration_mean           21             2        0.0952      2024-02-20T17:12:00Z   0.010377    0.009999
433  ts-preserve-other-service              trace.to_ts-user-service.duration_p95           21             2        0.0952      2024-02-20T17:12:00Z   0.023700    0.015765
525            ts-seat-service  trace.from_ts-preserve-other-service.duration_p95           21             2        0.0952      2024-02-20T17:06:00Z   0.437964    0.417707
608         ts-station-service  trace.from_ts-preserve-other-service.duration_...           21             2        0.0952      2024-02-20T17:02:00Z   0.003783    0.003579
609         ts-station-service  trace.from_ts-preserve-other-service.duration_p95           21             2        0.0952      2024-02-20T16:52:00Z   0.006862    0.006209
624      ts-ticketinfo-service  trace.from_ts-preserve-other-service.duration_...           21             2        0.0952      2024-02-20T17:09:00Z   0.357564    0.211772
625      ts-ticketinfo-service  trace.from_ts-preserve-other-service.duration_p95           21             2        0.0952      2024-02-20T17:09:00Z   0.776073    0.481264
768            ts-user-service  trace.from_ts-preserve-other-service.duration_...           21             2        0.0952      2024-02-20T17:12:00Z   0.010377    0.009999
769            ts-user-service  trace.from_ts-preserve-other-service.duration_p95           21             2        0.0952      2024-02-20T17:12:00Z   0.023700    0.015765```
```

In [ ]:
# In[4]:

import numpy as np
# Reuse metric_df already loaded and with timestamp parsed to UTC

# 1) Compute global P95(value) for every (cmdb_id, kpi_name) using the full metric.csv series
global_p95_df = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'global_p95'})
)

# 2) Filter metric.csv to the incident window (UTC)
start = pd.to_datetime('2024-02-20 16:48:43', utc=True)
end = pd.to_datetime('2024-02-20 17:18:43', utc=True)
metric_window2 = metric_df[(metric_df['timestamp'] >= start) & (metric_df['timestamp'] <= end)].copy()

# 3) Focus on the two services
services = ["ts-ticketinfo-service", "ts-preserve-other-service"]
metric_window2 = metric_window2[metric_window2['cmdb_id'].isin(services)].copy()

# Merge global p95 (precomputed from full series) into the windowed rows
metric_window2 = metric_window2.merge(global_p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# Mark exceedances: value > global_p95 (strict)
metric_window2['exceed'] = metric_window2['value'] > metric_window2['global_p95']

# 4) Aggregate per kpi_name for the selected services
agg_services = (
    metric_window2
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .agg(
        group_count=('value', 'size'),
        exceed_count=('exceed', 'sum'),
        earliest_exceed_timestamp=('timestamp', lambda x: x[metric_window2.loc[x.index, 'exceed']].min() if (x.size>0 and metric_window2.loc[x.index, 'exceed'].any()) else pd.NaT),
        max_value=('value', 'max'),
        global_p95=('global_p95', 'first')
    )
    .reset_index()
)

# Compute exceed_ratio and severity_ratio; handle zero or NaN global_p95 safely
agg_services['exceed_ratio'] = agg_services['exceed_count'] / agg_services['group_count']
agg_services['severity_ratio'] = np.where(agg_services['global_p95'] > 0,
                                          agg_services['max_value'] / agg_services['global_p95'],
                                          np.nan)

# Format earliest_exceed_timestamp as ISO UTC string or None
agg_services['earliest_exceed_timestamp'] = agg_services['earliest_exceed_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
agg_services['earliest_exceed_timestamp'] = agg_services['earliest_exceed_timestamp'].where(~agg_services['earliest_exceed_timestamp'].isna(), None)

# Round numeric columns for compactness
agg_services['max_value'] = agg_services['max_value'].round(6)
agg_services['global_p95'] = agg_services['global_p95'].round(6)
agg_services['exceed_ratio'] = agg_services['exceed_ratio'].round(4)
agg_services['severity_ratio'] = agg_services['severity_ratio'].round(4)

# Exclude any groups with group_count == 0 (though grouping from metric_window2 should not produce them)
agg_services = agg_services[agg_services['group_count'] > 0]

# Order by severity_ratio desc and limit to 50 rows
service_kpi_summary = agg_services.sort_values('severity_ratio', ascending=False).loc[
    :, ['cmdb_id', 'kpi_name', 'group_count', 'exceed_count', 'exceed_ratio',
        'earliest_exceed_timestamp', 'max_value', 'global_p95', 'severity_ratio']
].head(50)

# Display the result (variable only)
service_kpi_summary

```
Out[4]:
```
summary = (
    "Summary for ts-preserve-other-service and ts-ticketinfo-service (incident window 2024-02-20 16:48:43 to 2024-02-20 17:18:43 UTC):\n\n"
    "Key findings:\n"
    "- ts-preserve-other-service shows the largest relative spikes vs its historical P95:\n"
    "  * workload: group_count=25, exceed_count=2 (8.0%), earliest_exceed=2024-02-20T17:11:00Z, "
    "max=0.264517, global_p95=0.190670, severity_ratio=1.3873\n"
    "  * latency-50: group_count=24, exceed_count=2 (8.33%), earliest_exceed=2024-02-20T17:10:00Z, "
    "max=1.472500, global_p95=1.156047, severity_ratio=1.2737\n"
    "  * cpu: group_count=25, exceed_count=2 (8.0%), earliest_exceed=2024-02-20T16:53:00Z, "
    "max=1.433306, global_p95=1.243147, severity_ratio=1.1530\n"
    "  * socket, diskio, mem and latency-90 also show small exceedances with severity_ratio just above 1.0.\n\n"
    "- ts-ticketinfo-service shows smaller but noticeable deviations:\n"
    "  * latency-90: group_count=25, exceed_count=2 (8.0%), earliest_exceed=2024-02-20T17:04:00Z, "
    "max=0.060392, global_p95=0.055349, severity_ratio=1.0911\n"
    "  * diskio, cpu, socket, workload, mem likewise have 2 exceedances each with severity_ratio ~1.00–1.05.\n\n"
    "Notes on magnitude and prevalence:\n"
    "- Exceedances are relatively rare in the window (mostly 2 points per KPI; exceed_ratio ≈ 0.08).\n"
    "- However, the top severity ratios (e.g., preserve workload 1.387) indicate notable spikes above historical P95 for preserve service.\n"
    "- error KPIs have global_p95 = 0 and no exceedances (severity undefined).\n\n"
    "Interpretation and recommended next steps:\n"
    "- ts-preserve-other-service is the primary concern: multiple KPIs (workload, latency) spiked above their global P95 around 17:10–17:11 UTC.\n"
    "- ts-ticketinfo-service shows secondary, smaller spikes in latency and resource KPIs around ~17:04–17:14 UTC.\n"
    "- Investigate ts-preserve-other-service first: check its logs, incoming request patterns, and resource utilization around 17:05–17:12 UTC, and examine its calls to ticketinfo and order-other (trace data showed correlated trace exceedances).\n"
    "- Correlate these metric spikes with trace exceedances and logs to identify root cause (resource saturation, downstream latency, or network issues).\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  group_count  exceed_count  exceed_ratio earliest_exceed_timestamp     max_value    global_p95  severity_ratio
7   ts-preserve-other-service    workload           25             2        0.0800      2024-02-20T17:11:00Z  2.645170e-01  1.906700e-01          1.3873
3   ts-preserve-other-service  latency-50           24             2        0.0833      2024-02-20T17:10:00Z  1.472500e+00  1.156047e+00          1.2737
0   ts-preserve-other-service         cpu           25             2        0.0800      2024-02-20T16:53:00Z  1.433306e+00  1.243147e+00          1.1530
12      ts-ticketinfo-service  latency-90           25             2        0.0800      2024-02-20T17:04:00Z  6.039200e-02  5.534900e-02          1.0911
6   ts-preserve-other-service      socket           25             2        0.0800      2024-02-20T17:05:00Z  1.025000e+01  9.753333e+00          1.0509
9       ts-ticketinfo-service      diskio           25             2        0.0800      2024-02-20T16:54:00Z  6.551458e+04  6.251386e+04          1.0480
4   ts-preserve-other-service  latency-90           24             2        0.0833      2024-02-20T17:10:00Z  2.269500e+00  2.177051e+00          1.0425
1   ts-preserve-other-service      diskio           25             2        0.0800      2024-02-20T16:58:00Z  6.175151e+04  5.931797e+04          1.0410
14      ts-ticketinfo-service      socket           25             2        0.0800      2024-02-20T17:08:00Z  1.060000e+01  1.034333e+01          1.0248
8       ts-ticketinfo-service         cpu           25             2        0.0800      2024-02-20T16:51:00Z  6.880430e+00  6.753612e+00          1.0188
15      ts-ticketinfo-service    workload           25             2        0.0800      2024-02-20T16:52:00Z  4.196772e+01  4.136901e+01          1.0145
5   ts-preserve-other-service         mem           25             2        0.0800      2024-02-20T17:06:00Z  2.309855e+08  2.308913e+08          1.0004
11      ts-ticketinfo-service  latency-50           25             2        0.0800      2024-02-20T17:08:00Z  1.875500e-02  1.874700e-02          1.0004
13      ts-ticketinfo-service         mem           25             2        0.0800      2024-02-20T17:14:00Z  2.498729e+08  2.498285e+08          1.0002
2   ts-preserve-other-service       error           25             0        0.0000                      None  0.000000e+00  0.000000e+00             NaN
10      ts-ticketinfo-service       error           25             0        0.0000                      None  0.000000e+00  0.000000e+00             NaN```
```

In [ ]:
# In[5]:

# Reuse existing metric_df, trace_df, and start/end (incident window) from previous state

# Targets
target_services = ["ts-preserve-other-service", "ts-ticketinfo-service"]
target_kpis = ["workload","latency-50","cpu","socket","latency-90","mem","diskio"]

# 1) Compute global P95 for all (cmdb_id, kpi_name) using the full metric.csv series
metric_global_p95 = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'global_p95'})
)

# 2) Filter metric.csv to the incident window (UTC) - reuse start/end
metric_window = metric_df[(metric_df['timestamp'] >= start) & (metric_df['timestamp'] <= end)].copy()

# 3) Focus on the two services and selected KPIs
metric_target_window = metric_window[
    metric_window['cmdb_id'].isin(target_services) & metric_window['kpi_name'].isin(target_kpis)
].copy()

# Merge global p95 (precomputed from full series)
metric_target_window = metric_target_window.merge(metric_global_p95, on=['cmdb_id', 'kpi_name'], how='left')

# Keep only rows where value > global_p95 (strictly greater)
metric_exceeds = metric_target_window[metric_target_window['value'] > metric_target_window['global_p95']].copy()

# Format timestamp as ISO UTC string
metric_exceeds['timestamp_iso'] = metric_exceeds['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

# Select output columns and round numeric columns for compactness
metric_exceeds_out = metric_exceeds.loc[:, ['timestamp_iso', 'cmdb_id', 'kpi_name', 'value', 'global_p95']].copy()
metric_exceeds_out['value'] = metric_exceeds_out['value'].round(6)
metric_exceeds_out['global_p95'] = metric_exceeds_out['global_p95'].round(6)

# Limit to top 50 rows overall
metric_exceeds_head = metric_exceeds_out.head(50)

# Also compute count of returned rows per (cmdb_id, kpi_name) within these exceeds
metric_exceed_counts = (
    metric_exceeds_out
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .size()
    .rename(columns={'size': 'exceed_rows_count'})
    .sort_values(['exceed_rows_count', 'cmdb_id', 'kpi_name'], ascending=[False, True, True])
)

# --- Trace part ---

# Trace targets
trace_names = [
    "trace.to_ts-ticketinfo-service.duration_p95",
    "trace.from_ts-preserve-other-service.duration_p95",
    "trace.to_ts-ticketinfo-service.duration_mean",
    "trace.from_ts-preserve-other-service.duration_mean"
]

# 1) Compute global P95 for all (cmdb_id, trace_name) using full trace.csv
trace_global_p95 = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'global_p95'})
)

# 2) Filter trace.csv to the incident window
trace_window = trace_df[(trace_df['timestamp'] >= start) & (trace_df['timestamp'] <= end)].copy()

# 3) Filter to target trace_names
trace_target_window = trace_window[trace_window['trace_name'].isin(trace_names)].copy()

# Merge global p95 thresholds
trace_target_window = trace_target_window.merge(trace_global_p95, on=['cmdb_id', 'trace_name'], how='left')

# Keep only rows where value > global_p95 (strict)
trace_exceeds = trace_target_window[trace_target_window['value'] > trace_target_window['global_p95']].copy()

# Format timestamp as ISO UTC
trace_exceeds['timestamp_iso'] = trace_exceeds['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

# Select output columns and round numeric columns
trace_exceeds_out = trace_exceeds.loc[:, ['timestamp_iso', 'cmdb_id', 'trace_name', 'value', 'global_p95']].copy()
trace_exceeds_out['value'] = trace_exceeds_out['value'].round(6)
trace_exceeds_out['global_p95'] = trace_exceeds_out['global_p95'].round(6)

# Limit to top 50 rows
trace_exceeds_head = trace_exceeds_out.head(50)

# Counts per (cmdb_id, trace_name)
trace_exceed_counts = (
    trace_exceeds_out
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .size()
    .rename(columns={'size': 'exceed_rows_count'})
    .sort_values(['exceed_rows_count', 'cmdb_id', 'trace_name'], ascending=[False, True, True])
)

# Display compact results: metric exceeds head, metric counts, trace exceeds head, trace counts
metric_exceeds_head, metric_exceed_counts, trace_exceeds_head, trace_exceed_counts

```
Out[5]:
```
summary = (
    "Summary (incident window 2024-02-20 16:48:43 to 2024-02-20 17:18:43 UTC):\n\n"
    "Metric highlights (ts-preserve-other-service & ts-ticketinfo-service):\n"
    "- Total metric rows exceeding their group's global P95 in the window: 28 rows.\n"
    "- For each targeted KPI (workload, latency-50, cpu, socket, latency-90, mem, diskio) in both services, "
    "we observed 2 exceedance rows per (cmdb_id, kpi_name) during the window (i.e., each listed KPI had two points > global P95).\n"
    "- Example metric exceed timestamps span roughly 2024-02-20T16:51:00Z through 2024-02-20T17:15:00Z.\n"
    "- Notable metric spikes: ts-preserve-other-service workload (max 0.2645 vs global P95 0.1907), latency-50 (max 1.4725 vs P95 1.1560), "
    "and ts-ticketinfo-service cpu/diskio/mem showed modest exceedances (severity ratios ~1.02–1.09).\n\n"
    "Trace highlights (selected trace relations):\n"
    "- Total trace rows exceeding global P95 for the selected trace names in the window: 40 rows.\n"
    "- Many exceedances involve traces to/from ts-preserve-other-service (both \"to_...ticketinfo...\" and \"from_ts-preserve-other-service...\").\n"
    "- Exceedances are observed across multiple caller services (e.g., assurance, contacts, security, user, travel2, preserve itself) between ~16:57 and 17:15 UTC.\n"
    "- Example large trace deviation: preserve->ticketinfo duration_p95 reached ~0.776s vs a global P95 of ~0.481s (a meaningful latency jump).\n\n"
    "Overall interpretation and recommendation:\n"
    "- The metric and trace evidence point to ts-preserve-other-service as a focal point: its metrics (workload, latency) and many traces involving it exceeded historical P95 thresholds during the incident window.\n"
    "- ts-ticketinfo-service shows secondary, smaller metric spikes that are temporally nearby and are also implicated by trace relations with preserve.\n"
    "- Next steps: investigate ts-preserve-other-service (logs, request load, resource usage) around 17:05–17:12 UTC and correlate with the trace exceedances to identify whether preserve's resource pressure or downstream calls (ticketinfo, order-other) caused the observed latency spikes."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(            timestamp_iso                    cmdb_id    kpi_name         value    global_p95
7    2024-02-20T16:51:00Z      ts-ticketinfo-service         cpu  6.779265e+00  6.753612e+00
21   2024-02-20T16:52:00Z      ts-ticketinfo-service         cpu  6.880430e+00  6.753612e+00
27   2024-02-20T16:52:00Z      ts-ticketinfo-service    workload  4.158902e+01  4.136901e+01
28   2024-02-20T16:53:00Z  ts-preserve-other-service         cpu  1.357964e+00  1.243147e+00
50   2024-02-20T16:54:00Z      ts-ticketinfo-service      diskio  6.551458e+04  6.251386e+04
..                    ...                        ...         ...           ...           ...
298  2024-02-20T17:12:00Z  ts-preserve-other-service    workload  2.645170e-01  1.906700e-01
321  2024-02-20T17:14:00Z  ts-preserve-other-service      diskio  5.953279e+04  5.931797e+04
331  2024-02-20T17:14:00Z      ts-ticketinfo-service         mem  2.498729e+08  2.498285e+08
337  2024-02-20T17:15:00Z  ts-preserve-other-service  latency-90  2.179545e+00  2.177051e+00
345  2024-02-20T17:15:00Z      ts-ticketinfo-service         mem  2.498333e+08  2.498285e+08

[28 rows x 5 columns],                       cmdb_id    kpi_name  exceed_rows_count
0   ts-preserve-other-service         cpu                  2
1   ts-preserve-other-service      diskio                  2
2   ts-preserve-other-service  latency-50                  2
3   ts-preserve-other-service  latency-90                  2
4   ts-preserve-other-service         mem                  2
5   ts-preserve-other-service      socket                  2
6   ts-preserve-other-service    workload                  2
7       ts-ticketinfo-service         cpu                  2
8       ts-ticketinfo-service      diskio                  2
9       ts-ticketinfo-service  latency-50                  2
10      ts-ticketinfo-service  latency-90                  2
11      ts-ticketinfo-service         mem                  2
12      ts-ticketinfo-service      socket                  2
13      ts-ticketinfo-service    workload                  2,             timestamp_iso                    cmdb_id                                         trace_name     value  global_p95
166  2024-02-20T16:57:00Z       ts-assurance-service  trace.from_ts-preserve-other-service.duration_...  0.014403    0.011164
167  2024-02-20T16:57:00Z       ts-assurance-service  trace.from_ts-preserve-other-service.duration_p95  0.017799    0.011503
200  2024-02-20T16:58:00Z        ts-contacts-service  trace.from_ts-preserve-other-service.duration_...  0.019722    0.012614
201  2024-02-20T16:58:00Z        ts-contacts-service  trace.from_ts-preserve-other-service.duration_p95  0.035367    0.018313
314  2024-02-20T17:02:00Z        ts-security-service  trace.from_ts-preserve-other-service.duration_...  0.054733    0.042746
..                    ...                        ...                                                ...       ...         ...
600  2024-02-20T17:12:00Z            ts-user-service  trace.from_ts-preserve-other-service.duration_...  0.010377    0.009999
601  2024-02-20T17:12:00Z            ts-user-service  trace.from_ts-preserve-other-service.duration_p95  0.023700    0.015765
628  2024-02-20T17:13:00Z         ts-travel2-service  trace.from_ts-preserve-other-service.duration_...  0.744253    0.669301
672  2024-02-20T17:15:00Z  ts-preserve-other-service  trace.from_ts-preserve-other-service.duration_...  0.111444    0.099339
691  2024-02-20T17:15:00Z         ts-travel2-service  trace.from_ts-preserve-other-service.duration_p95  1.092938    1.088118

[40 rows x 5 columns],                   cmdb_id                                         trace_name  exceed_rows_count
12    ts-preserve-service       trace.to_ts-ticketinfo-service.duration_mean                  2
13    ts-preserve-service        trace.to_ts-ticketinfo-service.duration_p95                  2
22  ts-ticketinfo-service       trace.to_ts-ticketinfo-service.duration_mean                  2
23  ts-ticketinfo-service        trace.to_ts-ticketinfo-service.duration_p95                  2
24      ts-travel-service       trace.to_ts-ticketinfo-service.duration_mean                  2
..                    ...                                                ...                ...
21  ts-ticketinfo-service  trace.from_ts-preserve-other-service.duration_p95                  1
26     ts-travel2-service  trace.from_ts-preserve-other-service.duration_...                  1
27     ts-travel2-service  trace.from_ts-preserve-other-service.duration_p95                  1
30        ts-user-service  trace.from_ts-preserve-other-service.duration_...                  1
31        ts-user-service  trace.from_ts-preserve-other-service.duration_p95                  1

[32 rows x 3 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import numpy as np
import pandas as pd

# Reuse metric_df and start/end from previous state; ensure timestamps parsed
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Targets
candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]
# Incident window (reuse start/end if present)
start = pd.to_datetime('2024-02-20 16:48:43', utc=True)
end = pd.to_datetime('2024-02-20 17:18:43', utc=True)

# 1) Compute global P95(value) for every (cmdb_id, kpi_name) using the full file (must be done before filtering)
full_global_p95 = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'global_p95'})
)

# 2) Filter metric.csv to the incident window
metric_window = metric_df[(metric_df['timestamp'] >= start) & (metric_df['timestamp'] <= end)].copy()

# 3) Focus on candidate services and merge global_p95 (precomputed)
metric_window_cand = metric_window[metric_window['cmdb_id'].isin(candidates)].copy()
metric_window_cand = metric_window_cand.merge(full_global_p95, on=['cmdb_id', 'kpi_name'], how='left')

# 4) Aggregate per (cmdb_id, kpi_name)
def earliest_exceed_ts(group):
    exceed_idx = group['value'] > group['global_p95']
    if exceed_idx.any():
        return group.loc[exceed_idx, 'timestamp'].min()
    else:
        return pd.NaT

per_kpi = (
    metric_window_cand
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .apply(lambda g: pd.Series({
        'group_count': int(g.shape[0]),
        'exceed_count': int((g['value'] > g['global_p95']).sum()),
        'earliest_exceed_timestamp': earliest_exceed_ts(g),
        'max_value': float(g['value'].max()),
        'global_p95': float(g['global_p95'].iloc[0]) if not g['global_p95'].isna().all() else np.nan
    }))
    .reset_index()
)

# Compute severity_ratio and exceed_ratio
per_kpi['exceed_ratio'] = per_kpi['exceed_count'] / per_kpi['group_count']
per_kpi['severity_ratio'] = np.where(per_kpi['global_p95'] > 0,
                                     per_kpi['max_value'] / per_kpi['global_p95'],
                                     np.nan)

# Format earliest_exceed_timestamp as ISO UTC strings (None if NaT)
per_kpi['earliest_exceed_timestamp'] = per_kpi['earliest_exceed_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
per_kpi['earliest_exceed_timestamp'] = per_kpi['earliest_exceed_timestamp'].where(per_kpi['earliest_exceed_timestamp'].notna(), None)

# Round numeric columns for compact display
per_kpi[['max_value','global_p95','exceed_ratio','severity_ratio']] = per_kpi[['max_value','global_p95','exceed_ratio','severity_ratio']].round(6)

# Exclude groups with group_count == 0 (if any)
per_kpi = per_kpi[per_kpi['group_count'] > 0].copy()

# 5) Aggregate per cmdb_id (candidates)
agg_per_service = (
    per_kpi
    .groupby('cmdb_id', dropna=False)
    .agg(
        total_groups_with_data=('kpi_name', 'nunique'),
        total_exceed_count=('exceed_count', 'sum'),
        n_kpis_with_exceed=('exceed_count', lambda x: int((x>0).sum())),
        max_severity_ratio=('severity_ratio', lambda x: float(x.replace([np.inf, -np.inf], np.nan).max()) if len(x)>0 else np.nan),
        earliest_exceed_timestamp_overall=('earliest_exceed_timestamp', lambda s: (
            pd.to_datetime(s.dropna()).min().strftime('%Y-%m-%dT%H:%M:%SZ') if s.dropna().shape[0]>0 else None
        ))
    )
    .reset_index()
)

# Order aggregated table by total_exceed_count desc and limit to 10 rows
agg_summary = agg_per_service.sort_values('total_exceed_count', ascending=False).head(10)

# 6) Per-kpi rows for the top candidate by total_exceed_count
if agg_summary.shape[0] > 0:
    top_candidate = agg_summary.iloc[0]['cmdb_id']
    top_candidate_kpis = (
        per_kpi[per_kpi['cmdb_id'] == top_candidate]
        .sort_values('severity_ratio', ascending=False)
        .loc[:, ['cmdb_id','kpi_name','group_count','exceed_count','exceed_ratio','earliest_exceed_timestamp','max_value','global_p95','severity_ratio']]
        .head(20)
        .reset_index(drop=True)
    )
else:
    top_candidate = None
    top_candidate_kpis = per_kpi.head(0)

# Display compact results: aggregated summary and top candidate per-kpi rows
agg_summary, top_candidate_kpis

```
Out[6]:
```
summary = (
    "Incident window (UTC): 2024-02-20 16:48:43 to 2024-02-20 17:18:43.\n\n"
    "Aggregated findings for candidate services:\n"
    "- ts-travel-service (top by total exceedances):\n"
    "  * total_groups_with_data=8, total_exceed_count=16, n_kpis_with_exceed=8\n"
    "  * max_severity_ratio=~1.277 (earliest exceed at 2024-02-20T16:51:00Z)\n"
    "- ts-auth-service: total_groups_with_data=7, total_exceed_count=14, n_kpis_with_exceed=7, max_severity_ratio=~1.139\n"
    "- ts-order-service: total_groups_with_data=8, total_exceed_count=14, n_kpis_with_exceed=7, max_severity_ratio=~4.289\n"
    "- ts-route-service: total_groups_with_data=7, total_exceed_count=14, n_kpis_with_exceed=7, max_severity_ratio=~1.041\n"
    "- ts-train-service: total_groups_with_data=8, total_exceed_count=14, n_kpis_with_exceed=7, max_severity_ratio=~1.509\n\n"
    "Per-KPI details for the top candidate (ts-travel-service), ordered by severity_ratio:\n"
    "- error: group_count=25, exceed_count=2, earliest_exceed=2024-02-20T17:04:00Z, max=0.03330, global_p95=0.02608, severity_ratio≈1.277\n"
    "- workload: group_count=25, exceed_count=2, earliest_exceed=2024-02-20T16:51:00Z, max=15.28235, global_p95=12.93261, severity_ratio≈1.182\n"
    "- socket: group_count=25, exceed_count=2, earliest_exceed=2024-02-20T16:51:00Z, max=20.03333, global_p95=18.63804, severity_ratio≈1.075\n"
    "- latency-90: group_count=25, exceed_count=2, earliest_exceed=2024-02-20T17:05:00Z, max=0.546152, global_p95=0.531772, severity_ratio≈1.027\n"
    "- cpu: group_count=25, exceed_count=2, earliest_exceed=2024-02-20T16:54:00Z, max=4.834525, global_p95=4.786732, severity_ratio≈1.010\n"
    "- latency-50, diskio, mem: each has small exceedances with severity_ratio very close to 1.00.\n\n"
    "Interpretation and recommendations:\n"
    "- ts-travel-service is the primary candidate: it has the highest total number of metric exceedances (16) across 8 KPIs, with the earliest exceedance at ~16:51 UTC. Several KPIs (workload, error, socket, latency) show measurable spikes above their historical P95.\n"
    "- ts-order-service deserves attention too because, although it has fewer total exceedances (14), it contains the single largest severity spike (max_severity_ratio ≈ 4.29) which could indicate an isolated but significant issue in one KPI.\n"
    "- Other candidates (auth, route, train) all show multiple KPIs exceeding P95 as well (14 exceedances each) but with lower maximum severity ratios compared with order and travel.\n"
    "- Next steps: prioritize investigation of ts-travel-service (check logs, request load, resource metrics around 16:50–17:15 UTC), and concurrently inspect ts-order-service for the high-severity KPI to determine if that is a localized fault. Correlate with traces and logs to find root cause (resource saturation, downstream latency, or other errors)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id  total_groups_with_data  total_exceed_count  n_kpis_with_exceed  max_severity_ratio earliest_exceed_timestamp_overall
4  ts-travel-service                       8                  16                   8            1.277004              2024-02-20T16:51:00Z
0    ts-auth-service                       7                  14                   7            1.138605              2024-02-20T16:51:00Z
1   ts-order-service                       8                  14                   7            4.288554              2024-02-20T16:51:00Z
2   ts-route-service                       7                  14                   7            1.040869              2024-02-20T16:51:00Z
3   ts-train-service                       8                  14                   7            1.508997              2024-02-20T16:51:00Z,              cmdb_id    kpi_name  group_count  exceed_count  exceed_ratio earliest_exceed_timestamp     max_value    global_p95  severity_ratio
0  ts-travel-service       error           25             2          0.08      2024-02-20T17:04:00Z  3.330000e-02  2.607700e-02        1.277004
1  ts-travel-service    workload           25             2          0.08      2024-02-20T16:51:00Z  1.528235e+01  1.293261e+01        1.181691
2  ts-travel-service      socket           25             2          0.08      2024-02-20T16:51:00Z  2.003333e+01  1.863804e+01        1.074863
3  ts-travel-service  latency-90           25             2          0.08      2024-02-20T17:05:00Z  5.461520e-01  5.317720e-01        1.027042
4  ts-travel-service         cpu           25             2          0.08      2024-02-20T16:54:00Z  4.834525e+00  4.786732e+00        1.009985
5  ts-travel-service  latency-50           25             2          0.08      2024-02-20T17:07:00Z  2.006200e-02  1.988300e-02        1.009001
6  ts-travel-service      diskio           25             2          0.08      2024-02-20T16:55:00Z  6.497984e+04  6.470263e+04        1.004284
7  ts-travel-service         mem           25             2          0.08      2024-02-20T17:14:00Z  2.718726e+08  2.718595e+08        1.000048)```
```

In [ ]:
# In[7]:

import numpy as np
import pandas as pd

# Reuse trace_df and start/end if available; otherwise (safeguard) load and parse timestamps
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Candidate services and incident window (reuse start/end if present)
candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]
start = pd.to_datetime('2024-02-20 16:48:43', utc=True)
end = pd.to_datetime('2024-02-20 17:18:43', utc=True)

# 1) Compute global P95(value) per (cmdb_id, trace_name) using the full trace.csv series
trace_global_p95 = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'global_p95'})
)

# 2) Filter trace.csv to the incident window
trace_window = trace_df[(trace_df['timestamp'] >= start) & (trace_df['timestamp'] <= end)].copy()

# 3) Focus on candidate cmdb_ids
trace_window_cand = trace_window[trace_window['cmdb_id'].isin(candidates)].copy()

# Merge global_p95 (computed from full series) into the windowed rows
trace_window_cand = trace_window_cand.merge(trace_global_p95, on=['cmdb_id', 'trace_name'], how='left')

# Mark exceedances (strictly greater than global_p95)
trace_window_cand['exceed'] = trace_window_cand['value'] > trace_window_cand['global_p95']

# 4) Aggregate per (cmdb_id, trace_name)
def earliest_exceed(group):
    exc = group.loc[group['exceed'], 'timestamp']
    return exc.min() if not exc.empty else pd.NaT

per_trace = (
    trace_window_cand
    .groupby(['cmdb_id', 'trace_name'], dropna=False)
    .apply(lambda g: pd.Series({
        'group_count': int(g.shape[0]),
        'exceed_count': int(g['exceed'].sum()),
        'earliest_exceed_timestamp': earliest_exceed(g),
        'max_value': float(g['value'].max()),
        'global_p95': float(g['global_p95'].iloc[0]) if not g['global_p95'].isna().all() else np.nan
    }))
    .reset_index()
)

# Compute ratios
per_trace['exceed_ratio'] = per_trace['exceed_count'] / per_trace['group_count']
per_trace['severity_ratio'] = np.where(per_trace['global_p95'] > 0,
                                      per_trace['max_value'] / per_trace['global_p95'],
                                      np.nan)

# Keep datetime for aggregation, and also prepare ISO string for output
per_trace['earliest_exceed_timestamp_iso'] = per_trace['earliest_exceed_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
per_trace['earliest_exceed_timestamp_iso'] = per_trace['earliest_exceed_timestamp_iso'].where(per_trace['earliest_exceed_timestamp_iso'].notna(), None)

# Round numeric columns for compactness
per_trace[['max_value','global_p95','exceed_ratio','severity_ratio']] = per_trace[['max_value','global_p95','exceed_ratio','severity_ratio']].round(6)

# 5) Aggregate per candidate cmdb_id
agg_per_candidate = (
    per_trace
    .groupby('cmdb_id', dropna=False)
    .agg(
        total_trace_groups_with_data=('trace_name', 'nunique'),
        total_trace_exceed_count=('exceed_count', 'sum'),
        n_trace_names_with_exceed=('exceed_count', lambda s: int((s > 0).sum())),
        max_trace_severity_ratio=('severity_ratio', lambda s: float(s.replace([np.inf, -np.inf], np.nan).max()) if len(s)>0 else np.nan),
        earliest_trace_exceed_timestamp_overall=('earliest_exceed_timestamp', lambda s: (
            pd.to_datetime(s.dropna()).min().strftime('%Y-%m-%dT%H:%M:%SZ') if s.dropna().shape[0]>0 else None
        ))
    )
    .reset_index()
)

# Order aggregated summary by total_trace_exceed_count desc and limit to 5 rows
agg_summary = agg_per_candidate.sort_values('total_trace_exceed_count', ascending=False).head(5)

# 6) For the top candidate by total_trace_exceed_count, prepare per-trace_name rows
if agg_summary.shape[0] > 0:
    top_candidate = agg_summary.iloc[0]['cmdb_id']
    top_candidate_traces = (
        per_trace[per_trace['cmdb_id'] == top_candidate]
        .sort_values(['exceed_count', 'severity_ratio'], ascending=[False, False])
        .loc[:, ['cmdb_id', 'trace_name', 'group_count', 'exceed_count', 'exceed_ratio',
                 'earliest_exceed_timestamp_iso', 'max_value', 'global_p95', 'severity_ratio']]
        .rename(columns={'earliest_exceed_timestamp_iso': 'earliest_exceed_timestamp'})
        .head(20)
        .reset_index(drop=True)
    )
else:
    top_candidate = None
    top_candidate_traces = per_trace.head(0)

# Final compact outputs
agg_summary, top_candidate_traces

```
Out[7]:
```
summary = (
    "Incident window (UTC): 2024-02-20 16:48:43 to 2024-02-20 17:18:43.\n\n"
    "Key aggregated findings (trace.csv):\n"
    "- ts-travel-service is the top candidate: 48 trace relations with data, 71 total trace exceedances across 36 trace names,\n"
    "  max severity ratio ≈ 1.407 (earliest exceed at 2024-02-20T16:51:00Z). This indicates many traces involving ts-travel-service exceeded their historical P95s.\n"
    "- ts-order-service is the second candidate: 36 trace relations, 53 exceedances across 27 trace names, with the largest single severity ratio ≈ 2.212 (earliest exceed 2024-02-20T16:52:00Z).\n"
    "- ts-route-service, ts-train-service, and ts-auth-service also show multiple trace exceedances (30, 30, and 18 exceedances respectively), but with lower max severity ratios than order/travel.\n\n"
    "Per-trace highlights for the top candidate (ts-travel-service), ordered by severity (top examples):\n"
    "1) trace.from_ts-preserve-service.duration_p95 — group_count=25, exceed_count=2, earliest_exceed=2024-02-20T17:07:00Z, max=1.881146, global_p95=1.336796, severity≈1.407\n"
    "2) trace.to_ts-order-service.duration_mean — group_count=25, exceed_count=2, earliest_exceed=2024-02-20T17:12:00Z, max=0.010490, global_p95=0.008212, severity≈1.277\n"
    "3) trace.from_ts-admin-travel-service.duration_mean — group_count=25, exceed_count=2, earliest_exceed=2024-02-20T17:10:00Z, max=0.544309, global_p95=0.443268, severity≈1.228\n"
    "4) trace.from_ts-admin-travel-service.duration_p95 — group_count=25, exceed_count=2, earliest_exceed=2024-02-20T17:10:00Z, max=1.546330, global_p95=1.263541, severity≈1.224\n"
    "5) trace.to_ts-ticketinfo-service.duration_p95 — group_count=25, exceed_count=2, earliest_exceed=2024-02-20T17:05:00Z, max=0.221657, global_p95=0.183102, severity≈1.211\n\n"
    "Interpretation and recommended next steps:\n"
    "- The trace evidence points to ts-travel-service as a primary trace-level hotspot during the incident window, with many relations modestly above historical P95 and a few notable spikes (severity up to ~1.41).\n"
    "- ts-order-service shows fewer relations but a higher single-case severity; it should be checked for isolated high-impact traces.\n"
    "- Investigate ts-travel-service traces and related downstream/upstream services (preserve, order, admin-travel, ticketinfo, seat) around 16:50–17:15 UTC: check logs, recent deploys, request patterns, and resource metrics to find the root cause and whether the trace exceedances correlate with metric anomalies.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id  total_trace_groups_with_data  total_trace_exceed_count  n_trace_names_with_exceed  max_trace_severity_ratio earliest_trace_exceed_timestamp_overall
4  ts-travel-service                            48                        71                         36                  1.407205                    2024-02-20T16:51:00Z
1   ts-order-service                            36                        53                         27                  2.211743                    2024-02-20T16:52:00Z
2   ts-route-service                            20                        30                         15                  1.091551                    2024-02-20T16:52:00Z
3   ts-train-service                            24                        30                         15                  1.437018                    2024-02-20T16:52:00Z
0    ts-auth-service                            12                        18                          9                  1.247661                    2024-02-20T16:52:00Z,               cmdb_id                                        trace_name  group_count  exceed_count  exceed_ratio earliest_exceed_timestamp    max_value   global_p95  severity_ratio
0   ts-travel-service       trace.from_ts-preserve-service.duration_p95           25             2          0.08      2024-02-20T17:07:00Z     1.881146     1.336796        1.407205
1   ts-travel-service           trace.to_ts-order-service.duration_mean           25             2          0.08      2024-02-20T17:12:00Z     0.010490     0.008212        1.277477
2   ts-travel-service  trace.from_ts-admin-travel-service.duration_mean           25             2          0.08      2024-02-20T17:10:00Z     0.544309     0.443268        1.227947
3   ts-travel-service   trace.from_ts-admin-travel-service.duration_p95           25             2          0.08      2024-02-20T17:10:00Z     1.546330     1.263541        1.223807
4   ts-travel-service       trace.to_ts-ticketinfo-service.duration_p95           25             2          0.08      2024-02-20T17:05:00Z     0.221657     0.183102        1.210571
5   ts-travel-service          trace.from_ts-seat-service.duration_mean           25             2          0.08      2024-02-20T17:06:00Z     0.059614     0.053109        1.122486
6   ts-travel-service      trace.from_ts-admin-travel-service.row_count           25             2          0.08      2024-02-20T16:59:00Z    12.000000    10.800000        1.111111
7   ts-travel-service            trace.to_ts-route-service.duration_p95           25             2          0.08      2024-02-20T17:00:00Z     0.005061     0.004637        1.091551
8   ts-travel-service               trace.to_ts-order-service.row_count           25             2          0.08      2024-02-20T16:53:00Z    79.000000    73.400000        1.076294
9   ts-travel-service              trace.from_ts-food-service.row_count           25             2          0.08      2024-02-20T16:52:00Z    34.000000    31.600000        1.075949
10  ts-travel-service          trace.to_ts-ticketinfo-service.row_count           25             2          0.08      2024-02-20T16:53:00Z   471.000000   439.800000        1.070941
11  ts-travel-service                trace.to_ts-seat-service.row_count           25             2          0.08      2024-02-20T16:53:00Z   157.000000   146.800000        1.069482
12  ts-travel-service              trace.from_ts-seat-service.row_count           25             2          0.08      2024-02-20T16:53:00Z   320.000000   300.800000        1.063830
13  ts-travel-service                     trace.from_root.duration_mean           25             2          0.08      2024-02-20T17:04:00Z     1.342878     1.263968        1.062430
14  ts-travel-service               trace.to_ts-train-service.row_count           25             2          0.08      2024-02-20T16:53:00Z   288.000000   271.600000        1.060383
15  ts-travel-service            trace.to_ts-seat-service.duration_mean           25             2          0.08      2024-02-20T17:08:00Z     0.146550     0.138410        1.058813
16  ts-travel-service                      trace.from_root.duration_p95           25             2          0.08      2024-02-20T17:07:00Z     2.782862     2.629488        1.058329
17  ts-travel-service           trace.to_ts-train-service.duration_mean           25             2          0.08      2024-02-20T17:07:00Z     0.063092     0.059695        1.056902
18  ts-travel-service            trace.from_ts-travel-service.row_count           25             2          0.08      2024-02-20T16:53:00Z  2197.000000  2095.800000        1.048287
19  ts-travel-service              trace.to_ts-travel-service.row_count           25             2          0.08      2024-02-20T16:53:00Z  2197.000000  2095.800000        1.048287)```
```

In [ ]:
# In[8]:

# Reuse previously loaded DataFrames and time window variables where available.
import pandas as pd
import numpy as np

# Safeguard: ensure dataframes exist and timestamps parsed to UTC
try:
    log_df
except NameError:
    log_df = pd.read_csv("log.csv")
if log_df['timestamp'].dtype == 'int64' or log_df['timestamp'].dtype == 'float64' or log_df['timestamp'].dtype == 'object':
    log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)

try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
if metric_df['timestamp'].dtype == 'int64' or metric_df['timestamp'].dtype == 'float64' or metric_df['timestamp'].dtype == 'object':
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Incident window (UTC)
start = pd.to_datetime('2024-02-20 16:48:43', utc=True)
end = pd.to_datetime('2024-02-20 17:18:43', utc=True)

# Part 1: Logs
candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# Filter logs to window and candidate services
log_window = log_df[(log_df['timestamp'] >= start) & (log_df['timestamp'] <= end) & (log_df['cmdb_id'].isin(candidates))].copy()

# rows_in_window per candidate
rows_in_window = log_window.groupby('cmdb_id').size().rename('rows_in_window')

# sum_log_error_count: sum of value where log_name == 'log.error_count'
error_counts = (
    log_window[log_window['log_name'] == 'log.error_count']
    .groupby('cmdb_id')['value']
    .sum()
    .rename('sum_log_error_count')
)

# max_log_row_count: max value where log_name == 'log.row_count' if present
row_counts_max = (
    log_window[log_window['log_name'] == 'log.row_count']
    .groupby('cmdb_id')['value']
    .max()
    .rename('max_log_row_count')
)

# Combine into one compact table, ensure all candidates present
log_summary = (
    pd.concat([rows_in_window, error_counts, row_counts_max], axis=1)
    .reindex(candidates)  # keep the requested order; missing candidates will show NaN
    .fillna({'rows_in_window': 0, 'sum_log_error_count': 0})  # fill sensible defaults
    .reset_index()
    .rename(columns={'index': 'cmdb_id'})
)

# Ensure numeric types and round if needed
log_summary['rows_in_window'] = log_summary['rows_in_window'].astype(int)
log_summary['sum_log_error_count'] = log_summary['sum_log_error_count'].astype(float).round(6)
log_summary['max_log_row_count'] = log_summary['max_log_row_count'].astype(float).round(6)

# Also return any log rows for candidates in window where log_name == 'log.error_count' and value > 0
log_error_positive = log_window[(log_window['log_name'] == 'log.error_count') & (log_window['value'] > 0)].copy()
if not log_error_positive.empty:
    log_error_positive['timestamp_iso'] = log_error_positive['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    log_error_positive_out = log_error_positive.loc[:, ['timestamp_iso', 'cmdb_id', 'log_name', 'value']].head(20).reset_index(drop=True)
else:
    # empty DataFrame with expected columns
    log_error_positive_out = pd.DataFrame(columns=['timestamp_iso', 'cmdb_id', 'log_name', 'value'])

# Part 2: metric socket exceeds for ts-travel-service and ts-order-service
services_socket = ["ts-travel-service","ts-order-service"]
k = 'socket'

# Compute global P95 per (cmdb_id, kpi_name) from full series (must be global, pre-filter)
metric_global_p95 = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'global_p95'})
)

# Filter metric to incident window and target services/kpi
metric_window = metric_df[(metric_df['timestamp'] >= start) & (metric_df['timestamp'] <= end)].copy()
metric_socket_window = metric_window[
    (metric_window['cmdb_id'].isin(services_socket)) & (metric_window['kpi_name'] == k)
].merge(metric_global_p95, on=['cmdb_id', 'kpi_name'], how='left')

# Keep only rows where value > global_p95 (strict)
metric_socket_exceeds = metric_socket_window[metric_socket_window['value'] > metric_socket_window['global_p95']].copy()

# Format timestamp ISO and round numbers; limit to 20 rows
if not metric_socket_exceeds.empty:
    metric_socket_exceeds['timestamp_iso'] = metric_socket_exceeds['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    metric_socket_out = metric_socket_exceeds.loc[:, ['timestamp_iso', 'cmdb_id', 'kpi_name', 'value', 'global_p95']].copy()
    metric_socket_out['value'] = metric_socket_out['value'].round(6)
    metric_socket_out['global_p95'] = metric_socket_out['global_p95'].round(6)
    metric_socket_out = metric_socket_out.head(20).reset_index(drop=True)
else:
    metric_socket_out = pd.DataFrame(columns=['timestamp_iso','cmdb_id','kpi_name','value','global_p95'])

# Final compact outputs
log_summary, log_error_positive_out, metric_socket_out

```
Out[8]:
```
Summary (incident window 2024-02-20 16:48:43 — 2024-02-20 17:18:43 UTC)

1) Logs (per candidate)
- ts-auth-service: rows_in_window=50, sum_log_error_count=0.0, max_log_row_count=390.0  
- ts-order-service: rows_in_window=50, sum_log_error_count=0.0, max_log_row_count=683.0  
- ts-route-service: rows_in_window=50, sum_log_error_count=0.0, max_log_row_count=1241.0  
- ts-train-service: rows_in_window=2,  sum_log_error_count=0.0, max_log_row_count=166.0  
- ts-travel-service: rows_in_window=50, sum_log_error_count=0.0, max_log_row_count=2477.0  

- Any log.error_count > 0 in window: none (no positive log.error_count rows found for these candidates).

2) Metric: socket KPI exceedances (value > global P95) for ts-travel-service and ts-order-service
Returned rows (timestamp UTC, cmdb_id, kpi_name, value, global_p95) — limited to 20 rows (all shown here):
- 2024-02-20T16:51:00Z, ts-travel-service, socket, value=18.705882, global_p95=18.638039  
- 2024-02-20T17:01:00Z, ts-order-service,  socket, value=13.800000, global_p95=13.796667  
- 2024-02-20T17:07:00Z, ts-travel-service, socket, value=20.033333, global_p95=18.638039  
- 2024-02-20T17:10:00Z, ts-order-service,  socket, value=13.866667, global_p95=13.796667  

(Each service shows two socket exceedances in the window; values are slightly above their historical P95 thresholds.)

Quick interpretation
- No log.error_count activity recorded for these candidate services during the window, so no direct error-log signal.
- ts-travel-service has the highest observed log.row_count (peak 2477) and shows socket metric exceedances (two points, modestly above P95). ts-order-service also shows two socket exceedances.
- Socket metric exceedances are present but relatively small in number and magnitude (values just above P95). Correlate these socket spikes with the high row_count for ts-travel-service and any trace/latency spikes to determine if connection/socket pressure or increased traffic contributed to the incident.

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id  rows_in_window  sum_log_error_count  max_log_row_count
0    ts-auth-service              50                  0.0              390.0
1   ts-order-service              50                  0.0              683.0
2   ts-route-service              50                  0.0             1241.0
3   ts-train-service               2                  0.0              166.0
4  ts-travel-service              50                  0.0             2477.0, Empty DataFrame
Columns: [timestamp_iso, cmdb_id, log_name, value]
Index: [],           timestamp_iso            cmdb_id kpi_name      value  global_p95
0  2024-02-20T16:51:00Z  ts-travel-service   socket  18.705882   18.638039
1  2024-02-20T17:01:00Z   ts-order-service   socket  13.800000   13.796667
2  2024-02-20T17:07:00Z  ts-travel-service   socket  20.033333   18.638039
3  2024-02-20T17:10:00Z   ts-order-service   socket  13.866667   13.796667)```
```

In [ ]:
# In[9]:

import numpy as np
import pandas as pd

# Reuse metric_df and incident window (start/end) from previous state; parse timestamps if needed
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
if metric_df['timestamp'].dtype != 'datetime64[ns, UTC]':
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

start = pd.to_datetime('2024-02-20 16:48:43', utc=True)
end   = pd.to_datetime('2024-02-20 17:18:43', utc=True)

# 1) Compute global P95 per (cmdb_id, kpi_name) using the full metric.csv (global thresholds BEFORE filtering)
global_p95_df = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'global_p95'})
)

# 2) Filter metric.csv to the incident window
metric_window = metric_df[(metric_df['timestamp'] >= start) & (metric_df['timestamp'] <= end)].copy()

# 3) Focus on ts-order-service and merge global_p95 (precomputed)
service = "ts-order-service"
order_window = metric_window[metric_window['cmdb_id'] == service].copy()
order_window = order_window.merge(global_p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# 4A) Per-kpi aggregate for ts-order-service
def earliest_exceed_ts(g):
    mask = g['value'] > g['global_p95']
    if mask.any():
        return g.loc[mask, 'timestamp'].min()
    return pd.NaT

per_kpi = (
    order_window
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .apply(lambda g: pd.Series({
        'group_count': int(g.shape[0]),
        'exceed_count': int((g['value'] > g['global_p95']).sum()),
        'earliest_exceed_timestamp': earliest_exceed_ts(g),
        'max_value': float(g['value'].max()) if not g.empty else np.nan,
        'global_p95': float(g['global_p95'].iloc[0]) if not g['global_p95'].isna().all() else np.nan
    }))
    .reset_index()
)

# Compute ratios, format timestamps, handle division by zero
per_kpi['exceed_ratio'] = per_kpi['exceed_count'] / per_kpi['group_count']
per_kpi['severity_ratio'] = np.where(per_kpi['global_p95'] > 0,
                                     per_kpi['max_value'] / per_kpi['global_p95'],
                                     np.nan)

# Convert earliest_exceed_timestamp to ISO UTC strings or None
per_kpi['earliest_exceed_timestamp'] = per_kpi['earliest_exceed_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
per_kpi['earliest_exceed_timestamp'] = per_kpi['earliest_exceed_timestamp'].where(per_kpi['earliest_exceed_timestamp'].notna(), None)

# Round numeric columns for compact display
per_kpi[['max_value','global_p95','exceed_ratio','severity_ratio']] = per_kpi[['max_value','global_p95','exceed_ratio','severity_ratio']].round(6)

# Exclude any kpi with group_count == 0 (if present)
per_kpi = per_kpi[per_kpi['group_count'] > 0].copy()

# Order by severity_ratio desc and limit to 20 rows
per_kpi_summary = per_kpi.sort_values('severity_ratio', ascending=False).loc[
    :, ['cmdb_id','kpi_name','group_count','exceed_count','exceed_ratio','earliest_exceed_timestamp','max_value','global_p95','severity_ratio']
].head(20).reset_index(drop=True)

# 4B) Raw exceedance rows for ts-order-service in window where value > global_p95
raw_exceeds = order_window[order_window['value'] > order_window['global_p95']].copy()
if not raw_exceeds.empty:
    raw_exceeds['timestamp_iso'] = raw_exceeds['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    raw_exceeds_out = raw_exceeds.loc[:, ['timestamp_iso','cmdb_id','kpi_name','value','global_p95']].copy()
    raw_exceeds_out[['value','global_p95']] = raw_exceeds_out[['value','global_p95']].round(6)
    raw_exceeds_out = raw_exceeds_out.head(50).reset_index(drop=True)
else:
    raw_exceeds_out = pd.DataFrame(columns=['timestamp_iso','cmdb_id','kpi_name','value','global_p95'])

# Display the two compact results (variable names only)
per_kpi_summary, raw_exceeds_out

```
Out[9]:
```
Summary for ts-order-service (incident window 2024-02-20 16:48:43 — 2024-02-20 17:18:43 UTC)

Key per-KPI findings (ordered by severity_ratio):
- latency-90: group_count=25, exceed_count=2 (8%), earliest_exceed=2024-02-20T16:58:00Z, max=0.132161, global_p95=0.030817, severity_ratio≈4.29 — the largest breach by far.
- cpu: group_count=25, exceed_count=2, earliest_exceed=2024-02-20T17:12:00Z, max=4.236585, global_p95=3.196089, severity_ratio≈1.33.
- workload: group_count=25, exceed_count=2, earliest_exceed=2024-02-20T16:51:00Z, max=11.008, global_p95=8.890947, severity_ratio≈1.24.
- diskio: group_count=25, exceed_count=2, earliest_exceed=2024-02-20T16:51:00Z, max=69906.28, global_p95=60368.81, severity_ratio≈1.16.
- latency-50, socket, mem: each had 2 exceedances with severity ratios ≈1.00–1.04 (small breaches).
- error: 4 rows in window but no exceedances (global_p95 = 0).

Raw exceedance samples (value > global_p95) observed in the window (timestamps UTC):
- latency-90: 2024-02-20T16:58:00Z and 2024-02-20T16:59:00Z (max 0.132161 vs P95 0.030817).
- cpu: 2024-02-20T17:12:00Z (4.236585 vs P95 3.196089) and 2024-02-20T17:13:00Z (3.315506).
- workload: 2024-02-20T16:51:00Z (11.008) and 2024-02-20T17:01:00Z (8.9923).
- diskio: 2024-02-20T16:51:00Z and 2024-02-20T17:09:00Z.
- socket: 2024-02-20T17:01:00Z and 2024-02-20T17:10:00Z.
- mem: 2024-02-20T17:14:00Z and 2024-02-20T17:15:00Z.
(Full raw exceedance rows were returned; these are representative examples.)

Conclusion and next steps:
- The strongest breach for ts-order-service is latency-90 (severity ≈ 4.29), with two high-latency samples at ~16:58–16:59 UTC. This points to an actual latency spike, not just a small resource blip.
- Secondary concerns: modest CPU, workload, and disk I/O increases later in the window (some exceedances around 17:12–17:14 UTC).
- Recommended immediate actions: investigate request/trace details for the 16:58–16:59 latency spike (slow endpoints, downstream calls), check order-service logs around that time, and correlate with CPU/diskio/workload rises around 17:12–17:15 to see if they contributed or are subsequent effects.

The original code execution output of IPython Kernel is also provided below for reference:

(            cmdb_id    kpi_name  group_count  exceed_count  exceed_ratio earliest_exceed_timestamp     max_value    global_p95  severity_ratio
0  ts-order-service  latency-90           25             2          0.08      2024-02-20T16:58:00Z  1.321610e-01  3.081700e-02        4.288554
1  ts-order-service         cpu           25             2          0.08      2024-02-20T17:12:00Z  4.236585e+00  3.196089e+00        1.325553
2  ts-order-service    workload           25             2          0.08      2024-02-20T16:51:00Z  1.100800e+01  8.890947e+00        1.238113
3  ts-order-service      diskio           25             2          0.08      2024-02-20T16:51:00Z  6.990628e+04  6.036881e+04        1.157987
4  ts-order-service  latency-50           25             2          0.08      2024-02-20T17:02:00Z  1.072600e-02  1.036400e-02        1.035003
5  ts-order-service      socket           25             2          0.08      2024-02-20T17:01:00Z  1.386667e+01  1.379667e+01        1.005074
6  ts-order-service         mem           25             2          0.08      2024-02-20T17:14:00Z  2.684568e+08  2.681815e+08        1.001027
7  ts-order-service       error            4             0          0.00                      None  0.000000e+00  0.000000e+00             NaN,            timestamp_iso           cmdb_id    kpi_name         value    global_p95
0   2024-02-20T16:51:00Z  ts-order-service      diskio  6.990628e+04  6.036881e+04
1   2024-02-20T16:51:00Z  ts-order-service    workload  1.100800e+01  8.890947e+00
2   2024-02-20T16:58:00Z  ts-order-service  latency-90  3.139800e-02  3.081700e-02
3   2024-02-20T16:59:00Z  ts-order-service  latency-90  1.321610e-01  3.081700e-02
4   2024-02-20T17:01:00Z  ts-order-service      socket  1.380000e+01  1.379667e+01
5   2024-02-20T17:01:00Z  ts-order-service    workload  8.992283e+00  8.890947e+00
6   2024-02-20T17:02:00Z  ts-order-service  latency-50  1.072600e-02  1.036400e-02
7   2024-02-20T17:09:00Z  ts-order-service      diskio  6.066604e+04  6.036881e+04
8   2024-02-20T17:10:00Z  ts-order-service      socket  1.386667e+01  1.379667e+01
9   2024-02-20T17:12:00Z  ts-order-service         cpu  4.236585e+00  3.196089e+00
10  2024-02-20T17:13:00Z  ts-order-service         cpu  3.315506e+00  3.196089e+00
11  2024-02-20T17:14:00Z  ts-order-service         mem  2.682561e+08  2.681815e+08
12  2024-02-20T17:15:00Z  ts-order-service  latency-50  1.044900e-02  1.036400e-02
13  2024-02-20T17:15:00Z  ts-order-service         mem  2.684568e+08  2.681815e+08)```
```